# SMP và RAD với SAM 3 thật trên cả 12 video

**Cách dùng:** Runtime → Change runtime type → **T4 GPU**, rồi Runtime → **Run all**. Chỉ cần cho phép truy cập Google Drive (và nhập token Hugging Face nếu chưa lưu `HF_TOKEN` trong Secrets).

**Nếu phiên bị ngắt:** mở lại notebook và **Run all** lần nữa. Video đã xong có file trong `per_seq/` trên Drive nên được bỏ qua. Điểm rủi ro đã lưu nên không phải học lại mô hình. Chỉ mất video đang chạy dở. Nên để tab mở và máy không ngủ trong lúc chạy.

**Thiết lập** giống thí nghiệm chính và lần chạy 2 video trước:
- phát hiện oracle;
- mô hình rủi ro học ngoài tập, chia 5 phần theo video;
- RAD có cùng số lần kích hoạt với SMP τ = 0,05;
- SAM 3 fp16.

**Khác lần chạy trước:**
- **Thời gian tách theo từng cách.** Mỗi cách có bộ nhớ đệm SAM 3 riêng, nên thời gian của một cách là thời gian nó tự chạy một mình. Lần chạy trước dùng chung bộ nhớ đệm, nên cách chạy sau trông rẻ hơn thực tế.
- **Có ghi thời gian từng bước.** Gồm ByteTrack, tạo ứng viên và đặc trưng, chấm điểm rủi ro, cửa sổ SAM 3. Từ đó tính FPS cả chuỗi (chưa tính detector, vì phát hiện là oracle).

**Ước tính thời gian** cho 10 video còn lại: khoảng 7 giờ T4, sai số có thể ±30%. Con số này lấy số khung VOS oracle × 1,22 (tỉ lệ SAM 3/oracle đo trên 2 video) × 1,31 giây/khung. Thường phải chạy qua 2–3 phiên Colab, và mỗi phiên tốn thêm khoảng 15–25 phút cho cài đặt, tải ảnh, chạy ByteTrack và nạp mô hình.

**Kết quả** nằm trong `MyDrive/Saudh/computer_vision/rad-work/outputs/dispatch_sam3_all`:
- `per_seq/<video>.csv`: kết quả từng video;
- `all_per_seq.csv`: gộp mọi video;
- `summary.csv`, `summary.md`: tổng hợp, kèm khoảng bootstrap theo video.

## 1. Cấu hình

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess
CODE = '/content/rad_mot'                                        # mã nguồn, clone từ GitHub
WORK = '/content/drive/MyDrive/Saudh/computer_vision/rad-work'   # kết quả lưu trên Drive
DS   = '/content/datasets/FastTracker'                           # ảnh và nhãn: ổ tạm của phiên
DATA = f'{DS}/train'
OUT  = f'{WORK}/outputs/dispatch_sam3_all'
OLD  = f'{WORK}/outputs/dispatch_sam3/sweep_per_sequence.csv'    # 2 video đã chạy SAM 3 hôm 05/11
PREC = 'fp16'
POLICIES = 'learned:1.0 smp:0.05'      # thêm 'smp+birth:0.05' nếu muốn baseline mạnh (tốn thêm khoảng 60%)
RERUN_OLD = False                      # True: chạy lại 2 video cũ để có thời gian tách theo từng cách (+khoảng 35 phút)

# Video nhẹ chạy trước để có kết quả sớm; video đêm nhiều vật ở xa nặng nhất nên để cuối.
# Số trong ngoặc: (khung VOS oracle của SMP, của RAD), dùng để ước tính thời gian.
ORACLE = {
    'task_night_occlusion': (196, 66),
    'task_day_right_turn_occlusion': (430, 228),
    'task_night_right_turn_pedestrian': (532, 233),
    'task_night_right_turn_pedestrian_occlusion': (258, 530),
    'task_night_occlusion_far_objects': (575, 228),
    'task_day_right_turn_pedestrian': (934, 278),
    'task_day_occlusion': (947, 631),
    'task_day_right_turn': (1910, 714),
    'task_day_pedestrian_crossing': (2009, 831),
    'task_night_far_objects': (2971, 1537),
    'task_tunnel': (150, 219),             # đã chạy hôm 05/11
    'task_day_left_turn': (593, 287),      # đã chạy hôm 05/11
}
OLD_SEQS = ['task_tunnel', 'task_day_left_turn']
QUEUE = [s for s in ORACLE if RERUN_OLD or s not in OLD_SEQS]
os.makedirs(f'{OUT}/per_seq', exist_ok=True)
todo = [s for s in QUEUE if not os.path.exists(f'{OUT}/per_seq/{s}.csv')]
print('đã xong:', [s for s in QUEUE if s not in todo])
print('còn lại:', todo)

## 2. GPU, mã nguồn, thư viện

In [ ]:
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'KHÔNG CÓ')
assert gpu, 'Chưa bật GPU: Runtime → Change runtime type → T4 GPU, rồi Run all lại.'
!rm -rf $CODE && git clone -q https://github.com/dien2810/rad_mot $CODE
%cd $CODE
!git log -1 --format='repo: commit %h, %s (%cr)'
!pip -q install -r requirements.txt huggingface_hub
if not os.path.isdir('/content/sam3'):
    !git clone -q https://github.com/facebookresearch/sam3 /content/sam3
!pip -q install -e /content/sam3

## 3. Script chạy (bản mới nhất, nhúng trong notebook)

Ghi đè `scripts/run_sam3_dispatch.py`, nên notebook không phụ thuộc việc repo đã push bản này hay chưa.

In [ ]:
%%writefile /content/rad_mot/scripts/run_sam3_dispatch.py
#!/usr/bin/env python
"""SMP and RAD with real SAM 3 on every video: one video at a time, resumable, timed.

Same setting as the oracle sweep of run_dispatch.py (oracle detections, out-of-fold risk
model, RAD budget b = 1, i.e. as many triggers as SMP at tau = 0.05). What it adds:

  * each policy gets its own SAM 3 cache, so a policy's time is what it would cost alone
    (run_dispatch.py shares one cache across policies, so the second policy looks cheaper);
  * wall-clock time per stage: ByteTrack, candidates + features, risk scoring, SAM 3 windows,
    and frames per second of the whole chain (detector excluded: detections are oracle);
  * results of a video go to <out>/per_seq/<seq>.csv as soon as it is done, and videos already
    there are skipped, so a dead Colab session only loses the video in progress;
  * out-of-fold risk scores are cached in <out>/risk_scores.csv.gz (trained once per output
    folder, reused after a restart).

Image frames (img1/) are needed only for the videos in --run; the risk model is trained on
the GT of every video under --data.

Example (Colab, T4):
    python scripts/run_sam3_dispatch.py --data /content/datasets/FastTracker/train \
        --precision fp16 --run task_night_occlusion task_day_right_turn_occlusion \
        --out /content/drive/MyDrive/.../outputs/dispatch_sam3_all
    python scripts/run_sam3_dispatch.py --out .../dispatch_sam3_all --summary-only \
        --old .../outputs/dispatch_sam3/sweep_per_sequence.csv
CPU dry run of the whole chain (oracle VOS instead of SAM 3, no frames needed):
    python scripts/run_sam3_dispatch.py --data ... --backend oracle --out /tmp/dry
"""
from __future__ import annotations

import argparse
import gc
import glob
import os
import pickle
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from rad.config import load_config  # noqa: E402
from rad.dispatch import policy as pol  # noqa: E402
from rad.dispatch.candidates import FEATURES, build_candidates  # noqa: E402
from rad.dispatch.model import fit_full  # noqa: E402
from rad.eval.idsw import evaluate  # noqa: E402
from rad.io.mot import discover_sequences  # noqa: E402
from rad.pipeline import load_detections, load_gt, make_backend, prepare, train_risk  # noqa: E402
from rad.sam.record import CachingBackend  # noqa: E402
from rad.sam.smp import SMPParams, run_smp  # noqa: E402
from rad.tracker.bytetrack import ByteTrackParams, run_bytetrack  # noqa: E402

SCORE = "p_y_risk"
ORDER = {"learned": 0, "smp": 1, "smp+birth": 2}  # RAD first: nothing in any cache yet


class CountingBackend:
    """Counts the frames the VOS model really produces (cache replays are not counted)."""

    def __init__(self, inner):
        self.inner = inner
        self.frames = 0

    def __getattr__(self, k):
        return getattr(self.inner, k)

    def propagate(self, start_frame, box_xyxy, max_len):
        for item in self.inner.propagate(start_frame, box_xyxy, max_len):
            self.frames += 1
            yield item


def parse_policies(items: list[str]) -> list[tuple[str, float]]:
    out = []
    for it in items:
        name, _, val = it.partition(":")
        if name not in ORDER:
            sys.exit(f"unknown policy {name!r} (use {', '.join(ORDER)})")
        out.append((name, float(val or (1.0 if name == "learned" else 0.05))))
    return sorted(out, key=lambda x: (ORDER[x[0]], x[1]))


def load_or_train_scores(data, out: str) -> dict:
    """Out-of-fold scores, cached per output folder; returns the model metrics."""
    path = os.path.join(out, "risk_scores.csv.gz")
    n = {sd.name: len(sd.cands) for sd in data}
    if os.path.exists(path):
        sc = pd.read_csv(path)
        got = sc.groupby("seq").size().to_dict()
        if got == n:
            for sd in data:
                sd.cands[SCORE] = sc.loc[sc.seq == sd.name, SCORE].to_numpy()
            print(f"[model] reused out-of-fold scores from {path}")
            return dict(reused=True)
        print("[model] cached scores do not match the candidates (different data or code): retraining")
    t = time.perf_counter()
    metrics = train_risk(data, "y_risk")
    metrics["train_seconds"] = round(time.perf_counter() - t, 1)
    pd.concat([pd.DataFrame({"seq": sd.name, SCORE: sd.cands[SCORE].to_numpy()}) for sd in data]
              ).to_csv(path, index=False)
    print("[model]", {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
    return metrics


def timing_model(data, out: str):
    """One model on all videos, used only to time risk scoring (scores come from the OOF models)."""
    path = os.path.join(out, "timing_model.pkl")
    if os.path.exists(path):
        with open(path, "rb") as fh:
            return pickle.load(fh)
    allc = pd.concat([sd.cands for sd in data], ignore_index=True)
    m = fit_full(allc, "y_risk")
    with open(path, "wb") as fh:
        pickle.dump(m, fh)
    return m


def time_cheap_stages(sd, cfg, model) -> dict:
    """Re-run ByteTrack and candidate building on this video, timed (outputs are identical)."""
    bt = ByteTrackParams(**{**asdict(cfg["bytetrack"]), "frame_rate": sd.seq.fps})
    t0 = time.perf_counter()
    tracks, log = run_bytetrack(sd.det, sd.length, bt)
    t1 = time.perf_counter()
    cands = build_candidates(log, tracks, cfg["candidates"])
    t2 = time.perf_counter()
    model.predict_proba(cands[list(FEATURES)].astype(float).to_numpy())
    t3 = time.perf_counter()
    if len(tracks) != len(sd.tracks) or len(cands) != len(sd.cands):
        print(f"[warn] {sd.name}: timed re-run differs from the prepared run (non-deterministic step?)")
    return dict(sec_bytetrack=t1 - t0, sec_candidates=t2 - t1, sec_score=t3 - t2)


def warm_up(sd, kw: dict) -> None:
    """Load SAM 3 and run a 3-frame window, so model loading is not billed to the first policy."""
    t = time.perf_counter()
    be = make_backend("sam3", sd, **{**kw, "verbose": False})
    f, _, x, y, w, h = sd.tracks[0, :6]  # frame, id, x, y, w, h of the first tracker box
    for _ in be.propagate(int(f), np.array([x, y, x + w, y + h]), 3):
        pass
    print(f"[sam3] model loaded and warmed up ({time.perf_counter() - t:.0f}s, not counted)", flush=True)


def run_video(sd, cfg, policies, backend: str, kw: dict, cheap: dict) -> list[dict]:
    p = SMPParams(**{**asdict(cfg["smp"]), "image_size": sd.size})
    fps_video = float(sd.seq.fps or 30)
    common = dict(seq=sd.name, frames=sd.length, video_fps=fps_video)
    base = evaluate(sd.gt, sd.tracks)
    rows = [dict(**common, policy="bytetrack", param=0.0, windows=0, vos_frames=0, model_frames=0, triggers=0,
                 idsw=base["idsw"], idf1=base["idf1"], mota=base["mota"], n_swap=0,
                 sec_bytetrack=cheap["sec_bytetrack"], sec_candidates=0.0, sec_score=0.0, sec_select=0.0,
                 sec_vos=0.0)]
    ref_n = len(pol.smp_baseline(sd.log, 0.05, p.gap_G))
    for name, param in policies:
        t0 = time.perf_counter()
        if name == "learned":
            trig = pol.budget(sd.cands.dropna(subset=[SCORE]), SCORE, int(round(param * max(ref_n, 1))))
        elif name == "smp":
            trig = pol.smp_baseline(sd.log, param, p.gap_G)
        else:
            trig = pol.smp_plus_births(sd.log, sd.cands, param, p.gap_G)
        t1 = time.perf_counter()
        counter = CountingBackend(make_backend(backend, sd, **kw))
        be = CachingBackend(counter, max_frames=4000) if backend == "sam3" else counter
        corrected, wins, stats = run_smp(sd.tracks, sd.log, be, trig, sd.length, p)
        t2 = time.perf_counter()
        ev = evaluate(sd.gt, corrected)
        uses_cands = name != "smp"
        rows.append(dict(**common, policy=name, param=param, windows=stats["windows"],
                         vos_frames=stats["vos_frames"], model_frames=counter.frames, triggers=len(trig),
                         idsw=ev["idsw"], idf1=ev["idf1"], mota=ev["mota"], n_swap=stats.get("n_SWAP", 0),
                         sec_bytetrack=cheap["sec_bytetrack"],
                         sec_candidates=cheap["sec_candidates"] if uses_cands else 0.0,
                         sec_score=cheap["sec_score"] if name == "learned" else 0.0,
                         sec_select=t1 - t0, sec_vos=t2 - t1))
        del be, counter, corrected, wins
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass
        r = rows[-1]
        print(f"[{sd.name}] {name}:{param:g}  triggers {r['triggers']}  windows {r['windows']}  "
              f"VOS frames {r['vos_frames']}  IDSW {base['idsw']} -> {r['idsw']}  "
              f"VOS time {r['sec_vos'] / 60:.1f} min", flush=True)
    for r in rows:
        r["sec_total"] = r["sec_bytetrack"] + r["sec_candidates"] + r["sec_score"] + r["sec_select"] + r["sec_vos"]
        r["fps"] = r["frames"] / r["sec_total"] if r["sec_total"] > 0 else np.nan
        r["x_realtime"] = r["sec_total"] / (r["frames"] / r["video_fps"])
    return rows


def bootstrap(per: pd.DataFrame, a: str, b: str, col: str, n_boot: int = 20000, seed: int = 0):
    """Ratio sum(b)/sum(a) over videos, with a 95% interval from resampling videos."""
    piv = per.pivot_table(index="seq", columns="policy", values=col, aggfunc="sum")
    if a not in piv or b not in piv:
        return None
    piv = piv[[a, b]].dropna()
    if len(piv) < 2 or piv[a].sum() == 0:
        return None
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(piv), size=(n_boot, len(piv)))
    va, vb = piv[a].to_numpy()[idx].sum(1), piv[b].to_numpy()[idx].sum(1)
    ok = va > 0
    r = vb[ok] / va[ok]
    return dict(ratio=float(piv[b].sum() / piv[a].sum()), lo=float(np.percentile(r, 2.5)),
                hi=float(np.percentile(r, 97.5)), videos=int(len(piv)))


def summarise(out: str, old: str | None = None) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(out, "per_seq", "*.csv")))
    if not files:
        sys.exit(f"no results in {out}/per_seq")
    new = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    new["source"] = "sam3_timed"
    per = new
    if old and os.path.exists(old):  # videos done earlier with run_dispatch.py (no per-policy timing)
        o = pd.read_csv(old)
        o = o[~o.seq.isin(new.seq.unique())].copy()
        if len(o):
            o["source"] = "sam3_old_untimed"
            per = pd.concat([new, o], ignore_index=True)
            print(f"[summary] + {o.seq.nunique()} video(s) from {old} (counts only, no timing)")
    per.to_csv(os.path.join(out, "all_per_seq.csv"), index=False)
    key = ["policy", "param"]
    g = per.groupby(key)
    summ = g[["windows", "vos_frames", "triggers", "idsw"]].sum()
    summ["videos"] = g["seq"].nunique()
    summ["idf1_mean"] = g["idf1"].mean()
    timed = per[per.source == "sam3_timed"].groupby(key)
    summ["timed_videos"] = timed["seq"].nunique()
    summ["timed_frames"] = timed["frames"].sum()
    summ["timed_video_minutes"] = timed.apply(lambda d: (d.frames / d.video_fps).sum() / 60)
    summ["sec_vos"] = timed["sec_vos"].sum()
    summ["sec_total"] = timed["sec_total"].sum()
    summ["fps"] = summ["timed_frames"] / summ["sec_total"]
    summ["x_realtime"] = summ["sec_total"] / (summ["timed_video_minutes"] * 60)
    summ["sec_per_vos_frame"] = summ["sec_vos"] / timed["vos_frames"].sum().replace(0, np.nan)
    summ = summ.reset_index()
    summ.to_csv(os.path.join(out, "summary.csv"), index=False)
    lines = ["# SAM 3 dispatch: summary", "",
             summ.round(3).to_string(index=False), ""]
    for col, label in (("vos_frames", "VOS frames RAD/SMP"), ("sec_total", "time RAD/SMP (timed videos)")):
        src = per if col == "vos_frames" else per[per.source == "sam3_timed"]
        bs = bootstrap(src, "smp", "learned", col)
        if bs:
            lines.append(f"{label}: {bs['ratio']:.2f} (95% bootstrap over {bs['videos']} videos: "
                         f"{bs['lo']:.2f}-{bs['hi']:.2f})")
    piv = per.pivot_table(index="seq", columns="policy", values="idsw", aggfunc="sum")
    if {"smp", "learned", "bytetrack"} <= set(piv.columns):
        d = (piv["smp"] - piv["learned"]).dropna()
        rng = np.random.default_rng(1)
        boots = d.to_numpy()[rng.integers(0, len(d), size=(20000, len(d)))].sum(1)
        lines.append(f"IDSW SMP - RAD: {d.sum():.0f} (95% bootstrap: {np.percentile(boots, 2.5):.0f} to "
                     f"{np.percentile(boots, 97.5):.0f}); ByteTrack {piv['bytetrack'].sum():.0f}, "
                     f"SMP {piv['smp'].sum():.0f}, RAD {piv['learned'].sum():.0f}")
    txt = "\n".join(lines)
    with open(os.path.join(out, "summary.md"), "w") as fh:
        fh.write(txt + "\n")
    print(txt)
    return summ


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", help="split folder with one sub-folder per video (gt/, seqinfo.ini, img1/)")
    ap.add_argument("--config", default=os.path.join(os.path.dirname(__file__), "..", "configs", "default.yaml"))
    ap.add_argument("--out", required=True)
    ap.add_argument("--run", nargs="*", default=None,
                    help="videos to run, in this order (default: every video with frames, or all for oracle)")
    ap.add_argument("--policies", nargs="*", default=["learned:1.0", "smp:0.05"],
                    help="policy:param, from learned (budget b), smp (tau), smp+birth (tau)")
    ap.add_argument("--backend", choices=["sam3", "oracle"], default="sam3")
    ap.add_argument("--precision", choices=["auto", "fp16"], default="fp16")
    ap.add_argument("--frames-on-gpu", action="store_true")
    ap.add_argument("--old", default=None, help="earlier sweep_per_sequence.csv to merge into the summary")
    ap.add_argument("--summary-only", action="store_true")
    args = ap.parse_args()
    os.makedirs(os.path.join(args.out, "per_seq"), exist_ok=True)
    if args.summary_only:
        summarise(args.out, args.old)
        return
    if not args.data:
        sys.exit("--data is required")
    cfg = load_config(args.config)
    policies = parse_policies(args.policies)
    seqs = discover_sequences(args.data)
    ign = dict(ignore_classes=cfg["gt_ignore_classes"], ignore_ioa=cfg["det_ignore_ioa"])
    t = time.perf_counter()
    data = []
    for s in seqs:
        _, gt_vis = load_gt(s, **ign)
        det = load_detections(s, "oracle", None, cfg["oracle_dets"], gt_vis=gt_vis, **ign)
        data.append(prepare(s, det, cfg["bytetrack"], cfg["candidates"], **ign))
    print(f"[prepare] {len(data)} videos, {sum(len(sd.cands) for sd in data):,} candidates "
          f"({time.perf_counter() - t:.0f}s)", flush=True)
    load_or_train_scores(data, args.out)
    tmodel = timing_model(data, args.out)
    by_name = {sd.name: sd for sd in data}
    order = args.run if args.run else [sd.name for sd in data]
    unknown = [s for s in order if s not in by_name]
    if unknown:
        sys.exit(f"unknown video(s): {unknown}; available: {sorted(by_name)}")
    kw = dict(cfg["oracle_backend"] if args.backend == "oracle" else cfg["sam3_backend"])
    if args.backend == "sam3":
        kw["precision"] = args.precision
        if args.frames_on_gpu:
            kw["offload_video_to_cpu"] = False
    done = {os.path.splitext(os.path.basename(f))[0] for f in glob.glob(os.path.join(args.out, "per_seq", "*.csv"))}
    todo = [n for n in order if n not in done and (args.backend == "oracle" or by_name[n].seq.frame_paths())]
    if args.backend == "sam3" and todo:
        warm_up(by_name[todo[0]], kw)
    for name in order:
        sd = by_name[name]
        if name in done:
            print(f"[skip] {name}: already in {args.out}/per_seq")
            continue
        if args.backend == "sam3" and not sd.seq.frame_paths():
            print(f"[skip] {name}: no img1/ frames")
            continue
        t = time.perf_counter()
        cheap = time_cheap_stages(sd, cfg, tmodel)
        rows = run_video(sd, cfg, policies, args.backend, kw, cheap)
        pd.DataFrame(rows).to_csv(os.path.join(args.out, "per_seq", f"{name}.csv"), index=False)
        print(f"[done] {name} in {(time.perf_counter() - t) / 60:.1f} min -> per_seq/{name}.csv", flush=True)
    summarise(args.out, args.old)


if __name__ == "__main__":
    main()

## 4. Token Hugging Face

Token dùng cho hai việc: tải bộ dữ liệu FastTracker và tải checkpoint SAM 3 (cần được cấp quyền truy cập trên Hugging Face). Lấy token theo thứ tự: đã đăng nhập, rồi `HF_TOKEN` trong Secrets của Colab, rồi mới hỏi.

In [ ]:
from huggingface_hub import get_token, login
if not get_token():
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not get_token():
    login()
print('token Hugging Face:', 'có' if get_token() else 'KHÔNG CÓ')

## 5. Tải nhãn của 12 video và ảnh của các video còn lại

Nhãn chỉ vài MB, cần cho cả 12 video để học mô hình rủi ro. Ảnh mỗi video 0,5–1,6 GB; script tải từng tệp zip, giải nén rồi xoá, nên ổ đĩa chỉ cần chỗ cho ảnh đã giải nén.

In [ ]:
!python scripts/download_fasttracker.py --out $DS --mode gt
need = [s for s in todo if not glob.glob(f'{DATA}/{s}/img1/*')]
print('cần tải ảnh:', need)
if need:
    !python scripts/download_fasttracker.py --out $DS --mode full --seqs {' '.join(need)}
missing = [s for s in todo if not glob.glob(f'{DATA}/{s}/img1/*')]
assert not missing, f'Chưa có ảnh: {missing}. Chạy lại ô này.'
!df -h /content | tail -1

## 6. Ước tính thời gian

In [ ]:
import pandas as pd
SEC_PER_ORACLE_FRAME = 1.22 * 1.31   # tỉ lệ SAM 3/oracle (2 video) × giây mỗi khung SAM 3 (T4, fp16)
est = pd.DataFrame([dict(video=s, khung_oracle=sum(ORACLE[s]), phut_uoc_tinh=round(sum(ORACLE[s]) * SEC_PER_ORACLE_FRAME / 60))
                    for s in todo])
if len(est):
    est['cong_don_gio'] = (est.phut_uoc_tinh.cumsum() / 60).round(1)
    display(est)
    print(f'Tổng: khoảng {est.phut_uoc_tinh.sum() / 60:.1f} giờ T4 (±30%), chưa tính 15–25 phút chuẩn bị mỗi phiên.')
else:
    print('Không còn video nào: ô sau chỉ tổng hợp kết quả.')

## 7. Chạy

Mỗi cửa sổ SAM 3 in một dòng (số khung, thời gian, RAM, GPU). Mỗi video xong in dòng `[done]` và ghi ngay `per_seq/<video>.csv` lên Drive.

Lần chạy đầu tiên trong thư mục kết quả sẽ học mô hình rủi ro (vài phút, chỉ dùng CPU) rồi lưu điểm vào `risk_scores.csv.gz`. Các phiên sau dùng lại điểm này.

In [ ]:
if todo:
    !python scripts/run_sam3_dispatch.py --data $DATA --precision $PREC --policies $POLICIES --run {' '.join(todo)} --out $OUT --old $OLD
else:
    !python scripts/run_sam3_dispatch.py --out $OUT --summary-only --old $OLD

## 8. Kết quả

- **FPS:** tính bằng số khung video chia cho tổng thời gian, gồm ByteTrack, tạo ứng viên và đặc trưng, chấm điểm (chỉ với RAD) và các cửa sổ SAM 3. **Chưa tính detector.**
- **`x_realtime`:** chậm hơn thời gian thực bao nhiêu lần.
- **Video chạy trước hôm 05/11:** chỉ có số khung và IDSW, không có thời gian, trừ khi đặt `RERUN_OLD = True`.

In [ ]:
from IPython.display import Markdown, display
per = pd.read_csv(f'{OUT}/all_per_seq.csv')
cols = ['seq', 'policy', 'triggers', 'windows', 'vos_frames', 'model_frames', 'idsw', 'sec_vos', 'sec_total', 'fps', 'x_realtime', 'source']
view = per[[c for c in cols if c in per]].copy()
for c in ('sec_vos', 'sec_total'):
    if c in view:
        view[c.replace('sec', 'phut')] = (view.pop(c) / 60).round(1)
display(view.round(2))
summ = pd.read_csv(f'{OUT}/summary.csv')
display(summ.round(3))
print(open(f'{OUT}/summary.md').read())
s = summ.set_index('policy')
if {'learned', 'smp'} <= set(s.index) and s.loc['learned', 'timed_videos'] > 0:
    display(Markdown(
        f"**Cho slide** ({int(s.loc['learned', 'timed_videos'])} video có đo thời gian, "
        f"{s.loc['learned', 'timed_video_minutes']:.0f} phút video, T4 fp16, chưa tính detector): "
        f"RAD {s.loc['learned', 'fps']:.1f} khung/giây (chậm {s.loc['learned', 'x_realtime']:.1f} lần thời gian thực), "
        f"SMP {s.loc['smp', 'fps']:.1f} khung/giây (chậm {s.loc['smp', 'x_realtime']:.1f} lần), "
        f"ByteTrack {s.loc['bytetrack', 'fps']:.0f} khung/giây."))